# Multilayer Perceptron (MLP)

A **multilayer perceptron (MLP)** is a feedforward neural network for tabular data. It stacks linear models with nonlinear activation functions, so the model can learn curved relationships instead of only straight-line effects.

In this notebook, we use the California Housing dataset to predict median house values from neighborhood-level features.


> *A multilayer perceptron is regression that learned to bend: stack linear layers with nonlinear hinges and almost any shape becomes drawable --- the art is stopping before it draws the noise.* --- Words of wisdom by Claude Fable 5 (2026)


## What Is an MLP?

For regression, an MLP takes a feature vector $x$ and produces a prediction $\hat y$. A two-hidden-layer MLP has the form

$$
h_1 = \operatorname{ReLU}(xW_1 + b_1), \qquad
h_2 = \operatorname{ReLU}(h_1W_2 + b_2), \qquad
\hat y = h_2W_3 + b_3.
$$

Each hidden layer first makes a linear combination of the previous layer, then applies a nonlinear activation. Without the nonlinear activation, stacking many linear layers would collapse back into one linear regression.


## How It Works

1. **Prepare the data.** Split the data into training, validation, and test sets. Standardize the features using the training set only.
2. **Define the network.** Choose the number of layers, the number of hidden units, and the activation function.
3. **Forward pass.** Feed a mini-batch of inputs through the network to get predictions.
4. **Loss.** Measure prediction error. For regression, a common choice is mean squared error.
5. **Backpropagation.** Compute the gradient of the loss with respect to every weight.
6. **Update.** Use an optimizer such as Adam or stochastic gradient descent to move the weights downhill.
7. **Validate and test.** Use validation loss to monitor training, then report performance once on the held-out test set.


## Why It Works

An MLP works because each layer creates new learned features from the previous layer. Early hidden units may respond to simple patterns in the inputs; later hidden units combine those patterns into more flexible predictors.

For tabular regression, the practical lesson is simple: an MLP can fit nonlinear structure, but it also has enough flexibility to overfit. That is why we keep a validation set, use standardized inputs, and evaluate only once on a final test set.


## Pros and Cons

**Pros**

- Learns nonlinear relationships without manually writing interaction terms.
- Works for regression and classification with small changes to the output layer and loss.
- PyTorch makes the training loop explicit, so students can see forward pass, loss, backpropagation, and update.

**Cons**

- More tuning choices than linear regression: architecture, learning rate, batch size, epochs, and regularization.
- Less interpretable than linear models or decision trees.
- Sensitive to feature scaling and train/test leakage.
- Can overfit if the network is too large or training runs too long.


## Setup and Data

The target in this dataset is median house value, measured in hundreds of thousands of dollars. We split the data into training, validation, and test sets. The scaler is fitted only on the training set, then reused for validation and test data.


In [2]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Load the California Housing data.
X, y = fetch_california_housing(return_X_y=True)

# Hold out the test set first; do not touch it during model selection.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=seed
)

# Split the remaining training data into training and validation sets.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=seed
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"training rows:   {X_train.shape[0]}")
print(f"validation rows: {X_val.shape[0]}")
print(f"test rows:       {X_test.shape[0]}")
print(f"features:        {X_train.shape[1]}")


training rows:   13209
validation rows: 3303
test rows:       4128
features:        8


## PyTorch Tensors and Mini-Batches

PyTorch models work with tensors. The `DataLoader` below shuffles the training rows and serves them in mini-batches. Mini-batches make each update cheaper and add useful randomness to gradient descent.


In [3]:
def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32)

X_train_t = to_tensor(X_train)
y_train_t = to_tensor(y_train).view(-1, 1)
X_val_t = to_tensor(X_val)
y_val_t = to_tensor(y_val).view(-1, 1)
X_test_t = to_tensor(X_test)
y_test_t = to_tensor(y_test).view(-1, 1)

batch_size = 128
generator = torch.Generator().manual_seed(seed)
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True,
    generator=generator,
)


## Define the MLP

This model has two hidden layers. ReLU supplies the nonlinearity; dropout randomly removes a small fraction of hidden activations during training, which helps reduce overfitting.


In [4]:
class HousingMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

model = HousingMLP(n_features=X_train.shape[1])
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)


HousingMLP(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)


## Train and Validate

This is the full training loop. The key PyTorch sequence is always the same: clear old gradients, run the forward pass, compute loss, call `backward()`, and update the parameters.

We keep the model state with the lowest validation loss. That makes the validation set a guardrail against training for too long.


In [5]:
num_epochs = 40
patience = 6
best_val_loss = float("inf")
best_state = None
stale_epochs = 0
history = []

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()       # clear old gradients
        preds = model(xb)           # forward pass
        loss = loss_fn(preds, yb)   # training loss
        loss.backward()             # backpropagation
        optimizer.step()            # parameter update
        running_loss += loss.item() * len(xb)

    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val_t), y_val_t).item()

    history.append({"epoch": epoch, "train_mse": train_loss, "val_mse": val_loss})

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d} | train MSE {train_loss:.4f} | val MSE {val_loss:.4f}")

    if stale_epochs >= patience:
        print(f"early stopping at epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history).tail()


epoch 01 | train MSE 2.6318 | val MSE 1.3119
epoch 05 | train MSE 0.4420 | val MSE 0.4210
epoch 10 | train MSE 0.3831 | val MSE 0.3795
epoch 15 | train MSE 0.3530 | val MSE 0.3542
epoch 20 | train MSE 0.3377 | val MSE 0.3443
epoch 25 | train MSE 0.3291 | val MSE 0.3301
epoch 30 | train MSE 0.3200 | val MSE 0.3197
epoch 35 | train MSE 0.3129 | val MSE 0.3273
epoch 40 | train MSE 0.3082 | val MSE 0.3185


,epoch,train_mse,val_mse
35,36,0.309767,0.310419
36,37,0.308588,0.322646
37,38,0.309620,0.336310
38,39,0.311706,0.320366
39,40,0.308237,0.318482


## Evaluate on the Test Set

The test set is used once, after training choices have been made. For California Housing, the target is in units of $100{,}000$, so an MAE of `0.38` means about $38{,}000$ in median house value.


In [6]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_t).numpy().ravel()

test_mse = mean_squared_error(y_test, y_pred)
test_rmse = float(np.sqrt(test_mse))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R^2:  {test_r2:.4f}")


Test MSE:  0.3076
Test RMSE: 0.5547
Test MAE:  0.3802
Test R^2:  0.7652


## Reading the Result

A lower MSE or MAE means better prediction accuracy on the held-out test set. The $R^2$ score compares the model to the simple baseline that always predicts the average training target.

The important teaching point is not the exact score. The important point is the workflow: split honestly, scale without leakage, define the network, train by backpropagation, monitor validation loss, and report test performance only at the end.
